# Apriori

## Importing the libraries

In [38]:
!pip install apyori


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [39]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

## Data Preprocessing

In [40]:
dataset = pd.read_csv('Market_Basket_Optimisation.csv', header = None)
transactions = dataset.apply(lambda row: row.dropna().astype(str).tolist(), axis=1).tolist()

# dataset.info()
# transactions = []
# # Method 1: Using len()
# total_rows = len(dataset)
# # Method 2: Using shape
# total_rows = dataset.shape[0]

# for i in range(total_rows):
#     transactions.append([str(dataset.values[i, j]) for j in range(20)])

```
transactions = dataset.apply(lambda row: row.dropna().astype(str).tolist(), axis=1).tolist()
```

#### Here's the breakdown of each part:
* dataset.apply(..., axis=1) — runs the lambda function on every row, one at a time. axis=1 means "go row by row" (the default axis=0 would go column by column).
* lambda row: — for each row, row is a pandas Series containing all the values in that row (e.g. ['bread', 'milk', NaN, NaN, ...]).
* row.dropna() — removes all NaN cells from that row. Since market basket data is sparse (most transactions have fewer than 20 items), this strips the empty padding.
* .astype(str) — converts every remaining value to a string. Important for Apriori libraries — they expect text, not mixed types.
* .tolist() (inner) — converts the pandas Series into a plain Python list: ['bread', 'milk', 'eggs'].
* .tolist() (outer) — after apply() returns a Series of lists, this final .tolist() wraps them all into one list-of-lists — your final transactions.

## Training the Apriori model on the dataset

In [41]:
from apyori import apriori
total_rows = len(dataset)
min_supp = (3*7) / total_rows
rules = apriori(transactions=transactions, min_support=min_supp, min_confidence=0.2, min_lift=3, min_length=2, max_length=2)

## Visualising the results

### Displaying the first results coming directly from the output of the apriori function

In [42]:
results = list(rules)
print(f"Total rules found: {len(results)}")
results

Total rules found: 10


[RelationRecord(items=frozenset({'chicken', 'extra dark chocolate'}), support=0.0027996267164378083, ordered_statistics=[OrderedStatistic(items_base=frozenset({'extra dark chocolate'}), items_add=frozenset({'chicken'}), confidence=0.23333333333333334, lift=3.8894074074074076)]),
 RelationRecord(items=frozenset({'chicken', 'light cream'}), support=0.004532728969470737, ordered_statistics=[OrderedStatistic(items_base=frozenset({'light cream'}), items_add=frozenset({'chicken'}), confidence=0.29059829059829057, lift=4.84395061728395)]),
 RelationRecord(items=frozenset({'escalope', 'mushroom cream sauce'}), support=0.005732568990801226, ordered_statistics=[OrderedStatistic(items_base=frozenset({'mushroom cream sauce'}), items_add=frozenset({'escalope'}), confidence=0.3006993006993007, lift=3.790832696715049)]),
 RelationRecord(items=frozenset({'pasta', 'escalope'}), support=0.005865884548726837, ordered_statistics=[OrderedStatistic(items_base=frozenset({'pasta'}), items_add=frozenset({'esca

### Putting the results well organised into a Pandas DataFrame

In [43]:
def inspect(results):
    lhs         = [tuple(result[2][0][0])[0] for result in results]
    rhs         = [tuple(result[2][0][1])[0] for result in results]
    supports    = [result[1] for result in results]
    confidences = [result[2][0][2] for result in results]
    lifts       = [result[2][0][3] for result in results]
    return list(zip(lhs, rhs, supports, confidences, lifts))
resultsinDataFrame = pd.DataFrame(inspect(results), columns = ['Left Hand Side', 'Right Hand Side', 'Support', 'Confidence', 'Lift'])

### Displaying the results non sorted

In [44]:
resultsinDataFrame

,Left Hand Side,Right Hand Side,Support,Confidence,Lift
0,extra dark chocolate,chicken,0.002800,0.233333,3.889407
1,light cream,chicken,0.004533,0.290598,4.843951
2,mushroom cream sauce,escalope,0.005733,0.300699,3.790833
3,pasta,escalope,0.005866,0.372881,4.700812
4,fromage blanc,honey,0.003333,0.245098,5.164271
5,herb & pepper,ground beef,0.015998,0.323450,3.291994
6,tomato sauce,ground beef,0.005333,0.377358,3.840659
7,light cream,olive oil,0.003200,0.205128,3.114710
8,whole wheat pasta,olive oil,0.007999,0.271493,4.122410
9,pasta,shrimp,0.005066,0.322034,4.506672


### Displaying the results sorted by descending lifts

In [49]:
# print(resultsinDataFrame.columns.tolist())
resultsinDataFrame = resultsinDataFrame.sort_values(by="Lift", ascending=False).reset_index(drop=True)
# resultsinDataFrame = resultsinDataFrame.nlargest(n=10, columns="Lift")
resultsinDataFrame


,Left Hand Side,Right Hand Side,Support,Confidence,Lift
0,fromage blanc,honey,0.003333,0.245098,5.164271
1,light cream,chicken,0.004533,0.290598,4.843951
2,pasta,escalope,0.005866,0.372881,4.700812
3,pasta,shrimp,0.005066,0.322034,4.506672
4,whole wheat pasta,olive oil,0.007999,0.271493,4.122410
5,extra dark chocolate,chicken,0.002800,0.233333,3.889407
6,tomato sauce,ground beef,0.005333,0.377358,3.840659
7,mushroom cream sauce,escalope,0.005733,0.300699,3.790833
8,herb & pepper,ground beef,0.015998,0.323450,3.291994
9,light cream,olive oil,0.003200,0.205128,3.114710
